In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from collections import defaultdict

import gc 
import glob

In [2]:
DATA_LOCATION = "../data/variants/baseline"

In [3]:
splits_df = pd.read_parquet(f"{DATA_LOCATION}/split_map.parquet.parquet") 
batch_files = glob.glob(f"{DATA_LOCATION}/batch_*.parquet")

In [ ]:
model = nn.Sequential(
    nn.Linear(2048, 512),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(128, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
epochs = 25
batch_size_nn = 128

print("Training started")

for epoch in range(epochs):
    epoch_train_loss = 0.0
    train_samples_count = 0
    
    epoch_val_loss = 0.0
    val_samples_count = 0
    
    for file_path in batch_files:
        df_batch = pd.read_parquet(file_path)
        df_batch = pd.merge(df_batch, splits_df, on='activity_id', how='inner')

        train_mask = df_batch['split'] == 'train'
        val_mask   = df_batch['split'] == 'val'
        
        df_train = df_batch[train_mask]
        df_val   = df_batch[val_mask]
        
        if len(df_train) > 0:
            X_np = np.stack(df_train["morgan_fp"].values).astype(np.float32)
            y_train = df_train['pic50'].values

            X_train_t = torch.from_numpy(X_np)
            y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
            
            loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size_nn, shuffle=True)
            
            model.train()
            for bX, by in loader:
                optimizer.zero_grad()
                preds = model(bX)
                loss = criterion(preds, by)
                loss.backward()
                optimizer.step()
                
                epoch_train_loss += loss.item() * len(bX)
                train_samples_count += len(bX)
                
        if len(df_val) > 0:
            X_val_np = np.stack(df_val['morgan_fp'].values).astype(np.float32)
            y_val = df_val['pic50'].values
            
            X_val_t = torch.from_numpy(X_val_np)
            y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
            
            model.eval()
            with torch.no_grad():
                val_preds = model(X_val_t)
                v_loss = criterion(val_preds, y_val_t)
                
                epoch_val_loss += v_loss.item() * len(X_val_t)
                val_samples_count += len(X_val_t)
        
        del df_batch, df_train, df_val
        if 'X_train' in locals(): del X_train, X_train_t, y_train, y_train_t, loader
        if 'X_val' in locals(): del X_val, X_val_t, y_val, y_val_t
        gc.collect() 

        print("single batch finished")

    avg_train_loss = epoch_train_loss / train_samples_count if train_samples_count > 0 else 0
    avg_val_loss = epoch_val_loss / val_samples_count if val_samples_count > 0 else 0
    
    print(f"Epoka [{epoch+1:3d}/{epochs}] | Train MSE: {avg_train_loss:.4f} | Val MSE: {avg_val_loss:.4f}")

print("\nTraining finished")

Training started
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
Epoka [  1/25] | Train MSE: 1.5500 | Val MSE: 1.3596
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
single batch finished
Epoka [  2/25] | Train MSE: 1.1668 | Val MSE: 1.16

KeyboardInterrupt: 

In [9]:
model.eval()

# 2. Listy na zebranie wszystkich predykcji i prawdziwych wartości
all_test_preds = []
all_test_true = []

with torch.no_grad():
    for file_path in batch_files:
        df_batch = pd.read_parquet(file_path)
        df_batch = pd.merge(df_batch, splits_df, on='activity_id', how='inner')
        
        df_test = df_batch[df_batch['split'] == 'test']
        
        if len(df_test) > 0:

            X_test_np = np.stack(df_test["morgan_fp"].values).astype(np.float32)
            y_test = df_test['pic50'].values
            
            X_test_t = torch.from_numpy(X_test_np)
            y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
            
            test_preds = model(X_test_t)
            
            all_test_preds.extend(test_preds.flatten().tolist())
            all_test_true.extend(y_test_t.flatten().tolist())
            
        del df_batch, df_test
        if 'X_test' in locals(): del X_test, X_test_t, y_test, y_test_t
        gc.collect()

if len(all_test_true) > 0:
    test_r2 = r2_score(all_test_true, all_test_preds)
    test_rmse = np.sqrt(mean_squared_error(all_test_true, all_test_preds))
    
    print(f"\nWyniki na zbiorze testowym ({len(all_test_true)} cząsteczek):")
    print(f"Ostateczne R2:   {test_r2:.3f}")
    print(f"Ostateczne RMSE: {test_rmse:.3f}")
else:
    print("\nUwaga: Nie znaleziono żadnych danych testowych w plikach!")


Wyniki na zbiorze testowym (96647 cząsteczek):
Ostateczne R2:   0.452
Ostateczne RMSE: 0.974
